# Estudio 03 — Entrenamiento: Backpropagation, GD e Hiperparámetros
**Diplomado Superior en Redes Neuronales Artificiales y Deep Learning**  
Módulo 4 | Diana Blanco

---
## 📐 Retropropagación (Backpropagation)

Backpropagation es el algoritmo que ajusta los pesos calculando el gradiente del error respecto a cada peso usando la **regla de la cadena**:

$$\frac{\partial L}{\partial w_i} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial z} \cdot \frac{\partial z}{\partial w_i}$$

Donde $z = \sum w_i x_i + b$ y $y = f(z)$. El gradiente se propaga desde la salida hacia atrás, capa por capa.

---
## 🏃 Gradiente Descendiente — Variantes

| Variante | Actualización | Ventaja | Desventaja |
|---|---|---|---|
| **Batch GD** | $w = w - \eta \cdot \nabla L(\text{todo})$ | Estable | Lento con datasets grandes |
| **SGD** | $w = w - \eta \cdot \nabla L(\text{1 muestra})$ | Rápido por paso | Muy ruidoso |
| **Mini-batch GD** | $w = w - \eta \cdot \nabla L(\text{batch})$ | Balance | Tamaño de batch es HP |
| **Adam** | RMSProp + Momentum | Adaptativo, robusto | Más memoria |

---
## ⚙️ Hiperparámetros Principales

| Hiperparámetro | Rango típico | Efecto si es muy alto/bajo |
|---|---|---|
| Learning Rate ($\eta$) | 0.1 a 0.0001 | Alto → diverge; Bajo → lento |
| Batch size | 16, 32, 64, 128 | Alto → estable pero mucha memoria |
| Épocas | 10-100 | Alto → overfitting; Bajo → underfitting |
| Capas ocultas | 1-5 | Muchas → overfitting, lento |
| Neuronas/capa | 32-512 | Más → más capacidad, más parámetros |
| Dropout | 0.2-0.5 | Alto → underfitting; Bajo → overfitting |

In [ ]:
# MACHOTE: Configuración inicial
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from modulo4_libreria import *

INFO = setup_completo()

In [ ]:
# MACHOTE: Dataset sintético para experimentos
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000, n_features=10, n_informative=8,
    n_redundant=2, n_classes=2, random_state=42
)

datos = train_val_test_split(X, y, test_size=0.2, val_size=0.1)
X_train, X_val, X_test = datos['X_train'], datos['X_val'], datos['X_test']
y_train, y_val, y_test = datos['y_train'], datos['y_val'], datos['y_test']

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
# MACHOTE: Comparar arquitectura PEQUEÑA vs GRANDE
modelo_pequeno = crear_mlp((10,), 2, capas_ocultas=[16], dropout=0.1)
modelo_grande   = crear_mlp((10,), 2, capas_ocultas=[256, 128, 64], dropout=0.3)

print("=== MODELO PEQUEÑO ===")
modelo_pequeno.summary()
print(f"\nTotal params: {modelo_pequeno.count_params()}")

print("\n=== MODELO GRANDE ===")
modelo_grande.summary()
print(f"\nTotal params: {modelo_grande.count_params()}")

In [ ]:
# MACHOTE: Entrenar ambos modelos y comparar
print("Entrenando modelo PEQUEÑO...")
hist_peq = compilar_y_entrenar(modelo_pequeno, X_train, y_train, X_val, y_val,
                                num_clases=2, lr=0.001, epochs=20, batch_size=32,
                                verbose=0, early_stop_paciencia=5)

print("Entrenando modelo GRANDE...")
hist_gde = compilar_y_entrenar(modelo_grande, X_train, y_train, X_val, y_val,
                                num_clases=2, lr=0.001, epochs=20, batch_size=32,
                                verbose=0, early_stop_paciencia=5)

# Comparar curvas de pérdida
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist_peq.history['val_loss'], label='Pequeño', marker='o')
plt.plot(hist_gde.history['val_loss'], label='Grande', marker='s')
plt.title('Comparación: Pérdida en Validación')
plt.xlabel('Época'); plt.ylabel('Val Loss'); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(hist_peq.history['val_accuracy'], label='Pequeño', marker='o')
plt.plot(hist_gde.history['val_accuracy'], label='Grande', marker='s')
plt.title('Comparación: Accuracy en Validación')
plt.xlabel('Época'); plt.ylabel('Val Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# MACHOTE: Comparar diferentes learning rates
lrs = [0.1, 0.001, 0.0001]
historicos_lr = {}

for lr in lrs:
    print(f"\n▶ Entrenando con lr = {lr}")
    modelo = crear_mlp((10,), 2, capas_ocultas=[64, 32], dropout=0.2)
    hist = compilar_y_entrenar(modelo, X_train, y_train, X_val, y_val,
                               num_clases=2, lr=lr, epochs=15, batch_size=32,
                               verbose=0, early_stop_paciencia=20)
    historicos_lr[f'lr={lr}'] = hist

# Graficar comparación
plt.figure(figsize=(10, 5))
for label, hist in historicos_lr.items():
    plt.plot(hist.history['val_loss'], label=label, marker='o')
plt.title('Efecto del Learning Rate en Validación')
plt.xlabel('Época'); plt.ylabel('Val Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
# MACHOTE: Demostrar efecto de Early Stopping
modelo_es = crear_mlp((10,), 2, capas_ocultas=[128, 64], dropout=0.3)

hist_es = compilar_y_entrenar(
    modelo_es, X_train, y_train, X_val, y_val,
    num_clases=2, lr=0.001, epochs=100, batch_size=32,
    early_stop_paciencia=5, verbose=0
)

epochs_reales = len(hist_es.history['loss'])
print(f"Épocas ejecutadas: {epochs_reales} de 100 (early stop detuvo)")
graficar_historia(hist_es, titulo="Early Stopping - Efecto")

In [ ]:
# MACHOTE: Keras Tuner (requiere instalación: pip install keras-tuner)
# Código de referencia — descomentar si tienes keras-tuner instalado

# import keras_tuner as kt
# 
# def construir_modelo(hp):
#     capas = hp.Int('capas', 1, 3)
#     neuronas = hp.Int('neuronas', 32, 256, step=32)
#     lr = hp.Float('lr', 1e-4, 1e-2, sampling='log')
#     dropout = hp.Float('dropout', 0.1, 0.5)
#     
#     modelo = crear_mlp((10,), 2, 
#         capas_ocultas=[neuronas]*capas, 
#         dropout=dropout)
#     modelo.compile(optimizer=Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
#     return modelo
# 
# tuner = kt.RandomSearch(construir_modelo, objective='val_accuracy', max_trials=10)
# tuner.search(X_train, y_train, validation_data=(X_val, y_val), epochs=10)
# mejores_hp = tuner.get_best_hyperparameters()[0]
# print(mejores_hp.values)

---
## 📋 Guía rápida de ajuste de hiperparámetros

1. **Learning Rate:** Empieza con 0.001 (Adam). Si la pérdida no baja, prueba 0.01; si diverge, baja a 0.0001
2. **Batch Size:** Usa 32 por defecto. 64 si el dataset es grande, 16 si es pequeño
3. **Capas ocultas:** Empieza con 2-3 capas. Aumenta si hay underfitting, reduce si hay overfitting
4. **Dropout:** 0.2-0.3 para redes pequeñas, 0.5 para redes grandes
5. **Early Stopping:** Siempre actívalo. Paciencia = 5-10 épocas
6. **ReduceLROnPlateau:** Reduce lr automáticamente cuando la pérdida se estanca

> ⚠️ **Regla de oro:** Primero haz que overfitee (modelo grande, sin regularización), luego agrega regularización gradualmente.

---
## ✏️ Ejercicios (Tarea)

1. Prueba con `lr=0.5`. ¿Qué ocurre con la pérdida?
2. Entrena sin early stopping (`early_stop_paciencia=100`), ¿el modelo overfitea?
3. Crea un modelo con 5 capas ocultas y 256 neuronas. ¿Cuántos parámetros tiene?
4. Prueba `batch_size=4` vs `batch_size=256`. ¿Cuál es más rápido? ¿Cuál da mejor loss?

In [ ]:
# TODO: Experimentos con hiperparámetros
# 1. Encuentra el mejor learning rate para este dataset probando [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001]
# 2. Prueba con batch_size=64 y batch_size=8, compara tiempos
# 3. ¿Cuál es la combinación óptima de dropout y capas?

# === ESCRIBE TU CÓDIGO AQUÍ ===



# ════════════════════════════════════